# DANDI 000004: AMulti-Database Demo

## Overview: Neuronal Electrophysiology Analysis Using [DANDI 000004](https://dandi.github.io/example-notebooks/#dandiset-000004)

Neuroscience is a vast field, subject to domain experts. As such, there are terabytes of publicly available datasets on the topic, running from literature review to experimental design and analysis. One such database containing these datasets is DANDI: Distributed Archives for Neurophysiology Data Integration. Considered the gold standard for electrophysiology, DANDI uses the .nwb format (Neurodata Without Borders, a proprietary data format, organized into “Dandisets”) that plays nicely with vector databases, graph databases, and tabular databases. We explain our proposed approach below, using Dandiset 000004, “A NWB-based dataset and processing pipeline of human single-neuron activity during a declarative memory task”.

More on the DANDI API: https://dandi.readthedocs.io/en/latest/modref/dandiapi.html

This notebook demonstrates querying data that was ingested from a single streamed NWB session
(DANDI dandiset 000004) into the following databases:

| Database | Type | Purpose |
|---|---|---|
| **Neo4j** | Graph | Brain hierarchy (Subject -> Session -> Neuron -> Region) |
| **PostgreSQL** | Tabular | Subjects, sessions, neurons, trials |

In [1]:
import os
from dotenv import load_dotenv

# sanity checks
load_dotenv()
# print(os.environ.get("PG_DSN"))
# print(os.environ.get("NEO4J_URI"))

True

In [2]:
# create tables from schema.sql
import psycopg, os
from dotenv import load_dotenv
load_dotenv()

schema_path = os.path.join(os.path.dirname(os.path.abspath('.')), 'experiments', 'database', 'schema.sql')
schema_path = 'schema.sql'

with open(schema_path) as f:
    sql = f.read()

with psycopg.connect(os.environ['PG_DSN']) as conn, conn.cursor() as cur:
    cur.execute(sql)
    conn.commit()

In [3]:
# Stream and ingest every NWB session from DANDI 000004 into Postgres + Neo4j.
# No files are downloaded — data is fetched on-demand via HTTP range requests.
# Already-ingested sessions are detected from the DB and skipped automatically.
%run -i "data/ingest.py"

Dandiset 000004: 87 NWB assets found, 0 already ingested.


--- sub-P10HMH/sub-P10HMH_ses-20060901_ecephys+image.nwb ---
Session: H10_7
[Postgres] Ingested H10_7: 38 neurons, 200 trials.
[Neo4j]   Ingested H10_7: 38 neurons.

--- sub-P14HMH/sub-P14HMH_ses-20070601_obj-7studd_ecephys+image.nwb ---
Session: H14_18
[Postgres] Ingested H14_18: 33 neurons, 150 trials.
[Neo4j]   Ingested H14_18: 33 neurons.

--- sub-P15HMH/sub-P15HMH_ses-20070901_obj-a5xz9n_ecephys+image.nwb ---
Session: H15_21
[Postgres] Ingested H15_21: 14 neurons, 150 trials.
[Neo4j]   Ingested H15_21: 14 neurons.

--- sub-P14HMH/sub-P14HMH_ses-20070601_obj-1t8wrd5_ecephys+image.nwb ---
Session: H14_17
[Postgres] Ingested H14_17: 23 neurons, 200 trials.
[Neo4j]   Ingested H14_17: 23 neurons.

--- sub-P11HMH/sub-P11HMH_ses-20061101_ecephys+image.nwb ---
Session: H11_9
[Postgres] Ingested H11_9: 25 neurons, 200 trials.
[Neo4j]   Ingested H11_9: 25 neurons.

--- sub-P15HMH/sub-P15HMH_ses-20070901_obj-bdd49u_ecephys+image.nwb

## Neo4j: Graph Queries

In [4]:
import importlib
import utils.neo4j
importlib.reload(utils.neo4j)
from utils.neo4j import *

Node breakdown:

```sh
MATCH (s:Subject)-[:HAS_SESSION]->(sess:Session)-[:HAS_NEURON]->(n:Neuron)-[:LOCATED_IN]->(r:BrainRegion)
RETURN s, sess, n, r
```

![graph_overview](img/graph_overview.jpg)

In [34]:
# graph node counts
summary = get_graph_summary()
print('Graph node counts:')
display(summary)

Graph node counts:


,labels,count
0,[Neuron],1839
1,[Session],87
2,[Subject],59
3,[BrainRegion],4


![graph_summary](img/graph_summary.jpg)

In [39]:
# list available brain regions
brain_regions = get_brain_regions()
print('Brain regions:')
display(brain_regions)

Brain regions:


,brain_region,n_neurons
0,Right Amygdala,577
1,Left Amygdala,494
2,Right Hippocampus,403
3,Left Hippocampus,365


![brain_regions](img/brain_regions.jpg)

In [52]:
# experiment flow
# subject, session, neuron, region in that order
experiment_flow = get_experiment_flow()
print('Experiment flow:')
display(experiment_flow)

Experiment flow:


,s,sess,n,r
0,{'subject_id': 'P10HMH'},"{'date': '2006-09-01 00:00:00-07:00', 'session...","{'db_id': 38, 'unit_index': 37, 'brain_region'...",{'name': 'Left Amygdala'}
1,{'subject_id': 'P10HMH'},"{'date': '2006-09-01 00:00:00-07:00', 'session...","{'db_id': 37, 'unit_index': 36, 'brain_region'...",{'name': 'Left Amygdala'}
2,{'subject_id': 'P10HMH'},"{'date': '2006-09-01 00:00:00-07:00', 'session...","{'db_id': 36, 'unit_index': 35, 'brain_region'...",{'name': 'Left Amygdala'}
3,{'subject_id': 'P10HMH'},"{'date': '2006-09-01 00:00:00-07:00', 'session...","{'db_id': 35, 'unit_index': 34, 'brain_region'...",{'name': 'Left Amygdala'}
4,{'subject_id': 'P10HMH'},"{'date': '2006-09-01 00:00:00-07:00', 'session...","{'db_id': 34, 'unit_index': 33, 'brain_region'...",{'name': 'Left Amygdala'}
...,...,...,...,...
95,{'subject_id': 'P14HMH'},"{'date': '2007-06-01 00:00:00-07:00', 'session...","{'db_id': 90, 'unit_index': 4, 'brain_region':...",{'name': 'Right Amygdala'}
96,{'subject_id': 'P14HMH'},"{'date': '2007-06-01 00:00:00-07:00', 'session...","{'db_id': 89, 'unit_index': 3, 'brain_region':...",{'name': 'Right Amygdala'}
97,{'subject_id': 'P14HMH'},"{'date': '2007-06-01 00:00:00-07:00', 'session...","{'db_id': 88, 'unit_index': 2, 'brain_region':...",{'name': 'Right Amygdala'}
98,{'subject_id': 'P14HMH'},"{'date': '2007-06-01 00:00:00-07:00', 'session...","{'db_id': 87, 'unit_index': 1, 'brain_region':...",{'name': 'Right Amygdala'}


![experiment_flow](img/experiment_flow.jpg)

In [5]:
neuron_clusters = get_neuron_clusters()
print('Neuron clusters:')
display(neuron_clusters)

Neuron clusters:


,sess,n,r
0,"{'date': '2008-06-01 00:00:00-07:00', 'session...","{'db_id': 266, 'unit_index': 30, 'brain_region...",{'name': 'Left Amygdala'}
1,"{'date': '2018-12-01 00:00:00-08:00', 'session...","{'db_id': 1864, 'unit_index': 2, 'brain_region...",{'name': 'Left Amygdala'}
2,"{'date': '2018-12-01 00:00:00-08:00', 'session...","{'db_id': 1863, 'unit_index': 1, 'brain_region...",{'name': 'Left Amygdala'}
3,"{'date': '2018-12-01 00:00:00-08:00', 'session...","{'db_id': 1862, 'unit_index': 0, 'brain_region...",{'name': 'Left Amygdala'}
4,"{'date': '2018-04-01 00:00:00-07:00', 'session...","{'db_id': 1836, 'unit_index': 0, 'brain_region...",{'name': 'Left Amygdala'}
...,...,...,...
95,"{'date': '2018-01-01 00:00:00-08:00', 'session...","{'db_id': 1537, 'unit_index': 7, 'brain_region...",{'name': 'Left Amygdala'}
96,"{'date': '2018-01-01 00:00:00-08:00', 'session...","{'db_id': 1536, 'unit_index': 6, 'brain_region...",{'name': 'Left Amygdala'}
97,"{'date': '2018-01-01 00:00:00-08:00', 'session...","{'db_id': 1535, 'unit_index': 5, 'brain_region...",{'name': 'Left Amygdala'}
98,"{'date': '2018-01-01 00:00:00-08:00', 'session...","{'db_id': 1534, 'unit_index': 4, 'brain_region...",{'name': 'Left Amygdala'}


![neuron_clusters](img/neuron_clusters.jpg)

In [3]:
multi_region_sessions = get_multi_region_sessions()
print('Multi-region sessions:')
display(multi_region_sessions)

Multi-region sessions:


,sess.session_id,regions,unitCount
0,CS29_69,"[Left Amygdala, Left Hippocampus, Right Hippoc...",64
1,CS33_76,"[Left Amygdala, Right Amygdala, Right Hippocam...",53
2,H27_58,"[Left Hippocampus, Right Hippocampus]",49
3,CS54_125,"[Left Amygdala, Right Amygdala, Left Hippocamp...",46
4,H28_48,"[Left Amygdala, Right Amygdala, Right Hippocam...",45
...,...,...,...
70,H47_92,"[Right Amygdala, Right Hippocampus]",7
71,CS53_123,"[Left Amygdala, Right Hippocampus]",6
72,T90_2003,"[Left Amygdala, Right Amygdala]",5
73,T88_2002,"[Left Hippocampus, Right Hippocampus]",4


![multi_region_sessions](img/multi_region_sessions.jpg)

## PostgreSQL: Neuron Firing Statistics

After getting the data from Neo4j, we can refine our queries (and questions) even more with additional Postgres transactions.

In [ ]:
# if you update any function in postgres.py
# re-run this cell to get the lastest version
from utils.postgres import *

In [ ]:
# session overview
sessions = get_session_summary()
print(f'Ingested sessions: {len(sessions)}')
display(sessions)

In [ ]:
# firing statistics for ALL regions ALL sessions
stats = firing_stats()
print('Firing statistics by brain region (all sessions):')
print(stats)

In [ ]:
# filter: firing stats only for HIPPOCAMPAL neurons
hipp_stats = firing_stats_hippocampus()
print('Firing statistics for Hippocampus neurons (all sessions):')
print(stats)